# Lumina Multimodel - Experiment Journey

Purpose: single source of truth for hypothesis, runs, results, and reproducible commands.

## Active Hypothesis

- Hybrid routing (confidence + router) is working.
- Bottleneck is generator answer quality, not routing.
- Improving generator training data/objective should raise aggregator F1/task score.

## Key Results Snapshot

- Filtered baseline best at 5k: F1 **0.277**, task **0.413**, success@0.7 **0.385**, abstain **0.105**.
- HQ-med A/B @3178 (A100):
  - `filtered_gpt2`: route 0.700, F1 0.018, task 0.037
  - `hq_stagea_gpt2_medium`: route 0.700, F1 0.032, task 0.048
- Hybrid routing proxy (isolated local env): confidence 0.473, router 0.448, hybrid 0.647.

## Canonical Repro Flow

1) Build/refresh dataset: `python data/ingest_hq_datasets.py --out datasets_hq_med ...`
2) Train generators: `bash tools/train_generators_hq_med.sh`
3) Build calibration: `bash tools/build_conf_calibration_hq_med.sh`
4) Evaluate aggregator: `bash tools/run_aggregator_hq_med_5000.sh`
5) Log results to `notes/experiment_log.md`.

In [ ]:
# Sanity: show env versions in lumina_multimodel/.venv
import transformers, huggingface_hub, torch
print('transformers', transformers.__version__)
print('huggingface_hub', huggingface_hub.__version__)
print('torch', torch.__version__)

## Cleanup Decision Log

See: `notes/cleanup_inventory_2026-02-20.md` for keep/archive mapping before any file moves.

## Rigorous Roadmap (Single Source)

Roadmap with falsifiable claims and phase gates lives at:

- `lumina_multimodel/notes/roadmap_multimodel.md`

Use that file as canonical planning source; use `notes/experiment_log.md` for run-by-run evidence.

## Latest Results (2026-02-24)

- Stagea recipe variants (859-sample fast slice):
  - v3 (`answer_max_tokens=24`, quality on): F1 0.026 / task 0.073 / abstain 0.095
  - v3 with `max_new_tokens=80`: F1 0.017 / task 0.055 (worse)
  - v4 (remove answer cap, quality on): F1 0.026 / task 0.071
  - v5 (remove quality weighting): F1 0.026 / task 0.068
- Data clean v1 + stagea_v6:
  - @859: F1 0.031 / task 0.072 / abstain 0.130
  - threshold sweep best task config (`a=0.45,m=0.05`): F1 0.029 / task 0.074 / abstain 0.042
  - 3178 confirm: F1 0.023 / task 0.040 (`a=0.45,m=0.05`)

**Current decision:** threshold/decode tuning is near saturation; move to data enrichment/distillation + domain rebalance.


## Latest Results (2026-02-27)

- Frozen curated dataset: `datasets_hq_v2_curated`
  - general train 120000 / val 5000
  - math train 10433 / val 549
  - code train 18918 / val 995
- H100 Stage A run `lumina-hq-v2-stagea-001` (`gpt2-medium`, 2 epochs, quality weighting):
  - route accuracy `0.577`
  - aggregation F1 `0.048`
  - aggregation task score `0.087`
  - abstain `0.111`
- Decision:
  - curated HQ v2 is the best current dataset version
  - data quality is now producing measurable gains
  - next step is threshold sweep on this checkpoint before another training change


### Threshold Sweep (H100 checkpoint, 2026-02-27)

- Sweep over `abstain in {0.35,0.45,0.55}` and `margin in {0.03,0.05}` on 3210 samples.
- Best task score: `0.090` at `abstain=0.35/0.45` (either margin).
- High-abstain setting (`0.55`) slightly raises F1 to `0.048` but drops task score to `0.087-0.088`.
- Operating choice for this checkpoint: `abstain=0.45`, `margin=0.03`.
